Below is **Step 1: Architecture + data strategy**, then **Step 2: NGFS data access options (including why `pyam` often breaks)**, and **Step 3: a concrete Python pipeline skeleton** you can extend.
We’ll stay focused on **macro-economic NGFS variables suitable for Irish mortgage stress testing**.

---

# 1️⃣ What You’re Building (Conceptually)

You want a **repeatable macro-economic scenario pipeline** that:

1. Pulls **NGFS scenario data** (existing + new)
2. Normalises it across:

   * Scenario families
   * Time horizons (short / medium / long)
   * Variables relevant for mortgage credit risk
3. Feeds into:

   * ICAAP stress testing engine
   * Climate overlays (PD/LGD/collateral paths)
4. Is **transparent and defensible** for validation & regulators

Think of the pipeline in **four layers**:

```
NGFS Source Data
      ↓
Scenario Harmonisation Layer
      ↓
Macro-Financial Variable Selection
      ↓
Stress Engine Inputs (Quarterly / Annual)
```

---

# 2️⃣ NGFS Scenario Landscape (Old vs New)

You already know this, but it matters for how we structure the data.

## Legacy NGFS buckets (used in prior ICAAPs)

* **Orderly**
* **Disorderly**
* **Too-little-too-late**
* **Hot House World**

## Updated NGFS (Phase IV / 2023–2024)

These **map imperfectly** to the old structure — important for governance.

| Old Bucket      | New Scenario                      | Key Feature              |
| --------------- | --------------------------------- | ------------------------ |
| Orderly         | **Highway to Paris**              | Early, smooth transition |
| Disorderly      | **Sudden Wake-Up Call**           | Abrupt policy shock      |
| Mixed           | **Diverging Realities**           | Regional fragmentation   |
| Hot House World | **Disasters & Policy Stagnation** | Severe physical risk     |

⚠️ These are **model-driven IAM outputs**, not historical econometrics — that will matter later for model risk discussion.

---

# 3️⃣ What Macro Data You Actually Need (Mortgage-Relevant)

For an **Irish mortgage portfolio**, regulators expect **macro → credit → collateral transmission**.

### Core macro variables (minimum viable set)

You should extract **at least**:

| Category | Variable           | Why it matters             |
| -------- | ------------------ | -------------------------- |
| Growth   | Real GDP           | Income stress, arrears     |
| Labour   | Unemployment rate  | PD driver                  |
| Prices   | CPI / Inflation    | Affordability              |
| Rates    | Short & long rates | Affordability, refinancing |
| Housing  | Real house prices  | LGD / collateral           |
| Energy   | Energy prices      | Disposable income          |
| Climate  | Carbon price       | Transition severity        |

NGFS gives you most of these **directly or via proxies**.

---

# 4️⃣ Why `pyam` Often Fails (and What To Do)

Your experience with `pyam` is very common.

### Why `pyam` breaks in Jupyter

* It relies on:

  * `ixmp`
  * Java / JVM
  * Specific IAMC data formats
* Version mismatches cause:

  * Kernel crashes
  * Silent import errors
* Overkill if you **just want macro series**

👉 **Conclusion:**
For ICAAP macro ingestion, **you do NOT need `pyam`**.

---

# 5️⃣ Recommended Data Access Strategy (Pragmatic)

### Option A (Recommended): NGFS CSVs / Excel exports

* NGFS publishes:

  * Scenario explorer downloads
  * IAM output tables
* Easy to version-control
* Transparent for auditors

### Option B: World Bank / OECD overlays

* Used to:

  * Anchor baselines
  * Adjust country-specific paths (Ireland vs EU average)

---

# 6️⃣ Python Stack You Should Use

Minimal, robust, validator-friendly:

```python
import pandas as pd
import numpy as np

from pathlib import Path
```

Optional but useful later:

```python
import matplotlib.pyplot as plt
import seaborn as sns
```

No exotic climate libraries **at this stage**.

---

# 7️⃣ Data Pipeline – Folder Structure

I strongly recommend this upfront:

```
climate_icaap/
│
├── data/
│   ├── raw/
│   │   ├── ngfs_orderly.csv
│   │   ├── ngfs_highway_to_paris.csv
│   │   └── ...
│   ├── processed/
│   └── mappings/
│       └── variable_map.csv
│
├── notebooks/
│   ├── 01_ingest_ngfs.ipynb
│   ├── 02_harmonise_scenarios.ipynb
│   └── 03_macro_selection.ipynb
│
└── src/
    └── ngfs_pipeline.py
```

This matters a lot for **model governance**.

---

# 8️⃣ First Python Step: NGFS Ingestion

Let’s assume a **generic NGFS CSV structure**:

```python
df = pd.read_csv("data/raw/ngfs_highway_to_paris.csv")

df.head()
```

Typical NGFS columns:

* `Model`
* `Scenario`
* `Region`
* `Variable`
* `Unit`
* `2020`, `2025`, `2030`, ..., `2050`

### Melt to long format (critical step)

```python
df_long = df.melt(
    id_vars=["Model", "Scenario", "Region", "Variable", "Unit"],
    var_name="Year",
    value_name="Value"
)

df_long["Year"] = df_long["Year"].astype(int)
```

---

# 9️⃣ Filter for Ireland (or closest proxy)

NGFS often has:

* `Ireland`
* `EU`
* `Western Europe`

Decision needs documenting.

```python
df_ie = df_long[df_long["Region"].isin(["Ireland", "EU"])]
```

---

# 🔟 Variable Mapping Layer (Very Important)

Create a **controlled mapping**:

```python
variable_map = {
    "GDP|Real": "real_gdp",
    "Unemployment Rate": "unemployment",
    "Consumer Price Index": "cpi",
    "Long-term interest rate": "lt_rate",
    "Residential property prices": "house_prices",
    "Carbon price": "carbon_price"
}

df_ie["macro_var"] = df_ie["Variable"].map(variable_map)
df_ie = df_ie.dropna(subset=["macro_var"])
```

This lets you **swap NGFS versions without breaking downstream models**.

---

# 1️⃣1️⃣ Scenario Normalisation

You’ll want consistent naming across old & new scenarios:

```python
scenario_map = {
    "Orderly": "orderly",
    "Highway to Paris": "orderly",
    "Sudden Wake-Up Call": "disorderly",
    "Too little too late": "disorderly",
    "Diverging Realities": "fragmented",
    "Hot House World": "physical",
    "Disasters and Policy Stagnation": "physical"
}

df_ie["scenario_group"] = df_ie["Scenario"].map(scenario_map)
```

This will help your manager later when comparing **severity & rationale**.

---

# 1️⃣2️⃣ Output for Stress Engine

Final output should look like:

| Year | scenario | macro_var | value |
| ---- | -------- | --------- | ----- |

```python
final_macro = (
    df_ie
    .groupby(["Year", "scenario_group", "macro_var"])["Value"]
    .mean()
    .reset_index()
)
```

This is **exactly** what your PD/LGD overlays will expect.

---

# 1️⃣3️⃣ Next Steps (Logical Progression)

Once this is stable, the next notebooks should cover:

1. **Short-term focus (2026–2030)** vs long-term narratives
2. **Severity metrics** (GDP drawdown, unemployment peak)
3. **Mapping macro → mortgage risk**
4. **Why a given scenario is ICAAP-appropriate**

---

## Where We Can Go Next

If you want, next we can:

* Fix `pyam` properly **or justify not using it**
* Identify **which NGFS variables regulators actually expect**
* Build **Ireland-specific overlays** on EU NGFS data
* Create **severity ranking metrics** for your manager’s question
* Start mapping **macro → PD/LGD/collateral haircuts**

Scenario explorer website:
https://iiasa.ac.at/models-tools-data/ngfs-iiasa-scenario-explorer

Absolutely — let’s start **step-by-step** with a **practical pipeline build-out** that you can test with real NGFS data.

Before we dive into Python code, here are **the sources you’ll use to get the scenario data**:

### 📌 Where to Download NGFS Scenario Data

1. **NGFS Scenario Explorer — the primary bulk data source**

   * You can download **bulk CSV or Excel exports** of the full transition scenario data from the Scenario Explorer hosted by IIASA. You need to log in as *Guest* and then use the **Downloads** tab to get macroeconomic time series in CSV (IAMC-style). ([ngfs.net][1])

2. **NGFS User Guide & Technical Documentation**

   * Includes **instruction on the formats**, variable definitions, regions, and documentation on how the variables are structured. ([ngfs.net][2])

3. (Optional) **NGFS CA Climate Impact Explorer**

   * Physical risk data (if you plan to incorporate that later). ([ngfs.net][1])

> 🧠 *Important:* you’ll start with the **macro-financial CSVs** containing variables like GDP, unemployment, CPI, etc. — those are essential for your ICAAP pipeline.

---

# ✅ Step 1 — Create the Folder Structure

Here’s a standard project layout you should create **locally** (e.g., in your Jupyter workspace or version-controlled project):

```
climate_icaap/
│
├── data/
│   ├── raw/
│   ├── processed/
│   └── mappings/
│
├── notebooks/
│   ├── 01_ingest_ngfs.ipynb
│   ├── 02_harmonise_scenarios.ipynb
│   └── 03_macro_selection.ipynb
│
├── src/
│   ├── __init__.py
│   └── ngfs_pipeline.py
│
├── requirements.txt
└── README.md
```

### Directory Purpose

* **data/raw/** → original downloaded CSVs (never overwritten)
* **data/processed/** → cleaned, harmonised data outputs
* **data/mappings/** → JSON/CSV maps for variable and scenario names
* **src/ngfs_pipeline.py** → code you’ll build next
* **notebooks/** → exploratory Jupyter notebooks testing each step
* **requirements.txt** → Python dependencies

---

# ✅ Step 2 — Define the Core Pipeline Structure

In your **src/ngfs_pipeline.py**, we’ll build a modular pipeline using functions and a simple class to encapsulate the workflow.

### 📌 First Draft of the Pipeline File

```python
# src/ngfs_pipeline.py

import pandas as pd
from pathlib import Path

class NGFSDataPipeline:
    def __init__(self, raw_dir: str, processed_dir: str, mappings_dir: str):
        self.raw_dir = Path(raw_dir)
        self.processed_dir = Path(processed_dir)
        self.mappings_dir = Path(mappings_dir)

        # Ensure processed path exists
        self.processed_dir.mkdir(parents=True, exist_ok=True)

    def list_raw_files(self):
        """Return list of raw CSV files."""
        return list(self.raw_dir.glob("*.csv"))

    def load_raw(self, filename: str) -> pd.DataFrame:
        """Load a raw NGFS CSV into a DataFrame."""
        path = self.raw_dir / filename
        return pd.read_csv(path)

    def melt_scenario(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Convert wide year columns into long format.

        Example input:
        Model | Scenario | Region | Variable | Unit | 2020 | 2025 | ...
        """
        id_vars = ["Model", "Scenario", "Region", "Variable", "Unit"]
        # find year columns dynamically
        year_cols = [c for c in df.columns if c.isnumeric()]

        df_long = df.melt(
            id_vars=id_vars,
            value_vars=year_cols,
            var_name="Year",
            value_name="Value"
        )

        df_long["Year"] = df_long["Year"].astype(int)
        return df_long

    def save_processed(self, df: pd.DataFrame, name: str):
        """Save the cleaned DataFrame to processed directory."""
        out_path = self.processed_dir / name
        df.to_csv(out_path, index=False)
        print(f"Saved: {out_path}")

    def run_ingest(self):
        """Ingest all raw CSVs and produce melted long outputs."""
        for raw_file in self.list_raw_files():
            df = self.load_raw(raw_file.name)
            df_long = self.melt_scenario(df)

            # Save the processed version
            out_name = f"long_{raw_file.name}"
            self.save_processed(df_long, out_name)
```

### 🧠 How to Use This

In your **01_ingest_ngfs.ipynb**:

```python
from src.ngfs_pipeline import NGFSDataPipeline

p = NGFSDataPipeline(
    raw_dir="data/raw",
    processed_dir="data/processed",
    mappings_dir="data/mappings"
)

p.run_ingest()
```

This **reads all raw CSVs** and writes **long-format CSVs**, which you’ll then harmonise and filter.

---

# ✅ Step 3 — Define Variable & Scenario Mappings

You want mappings under **data/mappings/variable_map.csv** for consistency.

Here’s a simple CSV you can create manually:

```
original,standard
GDP|Real,real_gdp
Unemployment Rate,unemployment
Consumer Price Index,cpi
Long term interest rate,long_rate
Residential property prices,house_prices
Carbon price,carbon_price
```

And load it in code:

```python
def load_mappings(self):
    path = self.mappings_dir / "variable_map.csv"
    return pd.read_csv(path)
```

---

# ✅ Step 4 — Add Harmonisation Functions

Extend `ngfs_pipeline.py`:

```python
    def harmonise_variables(self, df: pd.DataFrame, var_map: pd.DataFrame) -> pd.DataFrame:
        """
        Apply variable mapping for standard names.

        var_map should have columns: "original", "standard"
        """
        mapping = dict(zip(var_map["original"], var_map["standard"]))
        df["macro_var"] = df["Variable"].map(mapping)

        # drop unmapped variables
        return df.dropna(subset=["macro_var"])

    def harmonise_scenarios(self, df: pd.DataFrame, scen_map: dict) -> pd.DataFrame:
        """
        Harmonise scenario names to groups
        """
        df["scenario_group"] = df["Scenario"].map(scen_map)
        return df.dropna(subset=["scenario_group"])
```

And your `scenario_map` dictionary in a notebook:

```python
scenario_map = {
    "Orderly": "orderly",
    "Highway to Paris": "orderly",
    "Sudden Wake-Up Call": "disorderly",
    "Too-little-too-late": "disorderly",
    "Diverging realities": "fragmented",
    "Disasters and Policy Stagnation": "physical",
    "Hot House World": "physical"
}
```

---

# 📌 Step 5 — Test End-to-End in Notebook

**Notebook Steps:**

1. Download or place NGFS macro CSVs in **data/raw/**
2. Run `p.run_ingest()` to make long files
3. Load `long_*.csv` into a DataFrame
4. Load mapping CSV
5. Apply `harmonise_variables()`
6. Apply `harmonise_scenarios()`
7. Save final output to **data/processed/final_macro.csv**

---

## What You Have Now

✅ **Folder structure established**
✅ **Data ingestion class with functions**
✅ **Mapping strategy for variables & scenarios**
✅ **Testable pipeline skeleton**

---

## What’s Next (when ready)

Once this initial structure is tested with real CSVs:

✔ We’ll add **regional filters (Ireland / EU)**
✔ Add **time interpolation or baseline adjustments**
✔ Build **visualisations & severity comparisons**
✔ Connect to your **stress engine ingestion format**

---

If you want, I can now provide a **ready-to-download NGFS macro sample CSV** you can use to test this pipeline before integrating the full dataset — just say so!

[1]: https://www.ngfs.net/ngfs-scenarios-portal/data-resources?utm_source=chatgpt.com "NGFS Scenarios Portal"
[2]: https://www.ngfs.net/system/files/2025-01/NGFS%20User%20Guide%20for%20NGFS%20Scenario%20Data%20Access%20%28Updated%20for%20Phase%20V%29.pdf?utm_source=chatgpt.com "User guide: Access to"
